# Empty Jupyter Notebook

This is a fresh, empty Jupyter notebook ready for your analysis and experimentation.

## Getting Started

1. **Add cells**: Use the `+` button or `B` (below) / `A` (above) shortcuts
2. **Cell types**: 
   - **Code cells**: Execute Python code
   - **Markdown cells**: Write documentation and explanations
3. **Run cells**: Use `Shift + Enter` to execute

## Tips

- Use `Tab` for autocompletion
- Use `Shift + Tab` for function help
- Use `Ctrl + /` to comment/uncomment lines
- Use `Esc` then `A`/`B` to add cells above/below
- Use `Esc` then `DD` to delete cells

Happy coding! 🚀


In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/dwarak/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/dwarak/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [48]:
# Simple Contextual Compression with Cohere Reranking
import os
import cohere
from typing import List, Dict, Any
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

# Set up Cohere API key (replace with your actual key)
os.environ["COHERE_API_KEY"] = getpass.getpass("LangChain API Key:")

print("✅ Basic imports successful!")
print("Cohere package version:", cohere.__version__ if hasattr(cohere, '__version__') else "Unknown")


✅ Basic imports successful!
Cohere package version: 5.19.0


In [79]:
# Simple Cohere Reranker
class SimpleCohereReranker:
    """Simple reranker using Cohere's rerank API"""
    
    def __init__(self, model: str = "rerank-v3.5", top_k: int = 5):
        self.model = model
        self.top_k = top_k
        self.cohere_client = cohere.Client()
    
    def rerank(self, query: str, documents: List[str]) -> List[Dict[str, Any]]:
        """Rerank documents based on query relevance"""
        try:
            # Try different parameter names for different Cohere versions
            rerank_params = {
                "model": self.model,
                "query": query,
                "documents": documents
            }
            
            # Try top_n first (newer versions)
            try:
                rerank_params["top_n"] = self.top_k
                rerank_response = self.cohere_client.rerank(**rerank_params)
            except TypeError:
                # Fallback to top_k (older versions)
                rerank_params.pop("top_n", None)
                rerank_params["top_k"] = self.top_k
                rerank_response = self.cohere_client.rerank(**rerank_params)
            
            results = []
            for result in rerank_response.results:
                results.append({
                    'index': result.index,
                    'relevance_score': result.relevance_score,
                    'document': documents[result.index]
                })
            return results
            
        except Exception as e:
            print(f"Error in reranking: {e}")
            # Fallback: return original order
            return [{'index': i, 'relevance_score': 1.0, 'document': doc} 
                   for i, doc in enumerate(documents[:self.top_k])]

print("✅ SimpleCohereReranker class created!")


✅ SimpleCohereReranker class created!


In [80]:
# Simple Base Retriever and Compression Retriever
from pydantic import Field
from typing import Optional
import uuid

class SimpleRetriever(BaseRetriever):
    """Simple retriever with sample documents"""
    
    documents: List[Document] = Field(default_factory=lambda: [
        Document(page_content="Python is a high-level programming language known for its simplicity and readability.", metadata={"source": "python.txt"}),
        Document(page_content="Machine learning is a subset of artificial intelligence that focuses on algorithms.", metadata={"source": "ml.txt"}),
        Document(page_content="Natural language processing (NLP) is the field of AI that deals with human language.", metadata={"source": "nlp.txt"}),
        Document(page_content="Deep learning uses neural networks with multiple layers to solve complex problems.", metadata={"source": "deep_learning.txt"}),
        Document(page_content="Data science combines statistics, programming, and domain expertise to extract insights.", metadata={"source": "data_science.txt"}),
    ])
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        return self.documents

class SimpleCompressionRetriever(BaseRetriever):
    """Simple compression retriever that uses Cohere for reranking"""
    
    base_retriever: BaseRetriever = Field(description="Base retriever to get initial documents")
    reranker: SimpleCohereReranker = Field(description="Reranker to compress/rerank documents")
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        # Get documents from base retriever
        docs = self.base_retriever._get_relevant_documents(query)
        
        # Extract document texts
        doc_texts = [doc.page_content for doc in docs]
        
        # Rerank using Cohere
        reranked_results = self.reranker.rerank(query, doc_texts)
        
        # Create new documents with reranked order
        compressed_docs = []
        for result in reranked_results:
            original_doc = docs[result['index']]
            new_doc = Document(
                page_content=original_doc.page_content,
                metadata={
                    **original_doc.metadata,
                    'relevance_score': result['relevance_score']
                }
            )
            compressed_docs.append(new_doc)
        
        return compressed_docs
    
    # GraphAgentSystem compatibility methods
    async def search_documents(self, query: str, limit: int = 5) -> Dict[str, Any]:
        """Search documents and return in VectorDBService format for GraphAgentSystem compatibility"""
        try:
            docs = self._get_relevant_documents(query)
            documents = []
            for doc in docs[:limit]:
                documents.append({
                    "id": doc.metadata.get("id", str(uuid.uuid4())),
                    "content": doc.page_content,
                    "metadata": doc.metadata,
                    "score": doc.metadata.get("relevance_score", doc.metadata.get("score", 0.0))
                })
            return {"success": True, "documents": documents}
        except Exception as e:
            return {"success": False, "documents": [], "error": str(e)}
    
    async def initialize(self):
        """Initialize method for GraphAgentSystem compatibility"""
        try:
            # Initialize base retriever if it has an initialize method
            if hasattr(self.base_retriever, 'initialize'):
                if asyncio.iscoroutinefunction(self.base_retriever.initialize):
                    await self.base_retriever.initialize()
                else:
                    self.base_retriever.initialize()
            
            # Initialize reranker if it has an initialize method
            if hasattr(self.reranker, 'initialize'):
                if asyncio.iscoroutinefunction(self.reranker.initialize):
                    await self.reranker.initialize()
                else:
                    self.reranker.initialize()
                    
            print("✅ Compression retriever initialized successfully")
        except Exception as e:
            print(f"⚠️ Warning during compression retriever initialization: {e}")
            # Don't raise the error, just log it

print("✅ Simple retriever classes created with GraphAgentSystem compatibility!")


✅ Simple retriever classes created with GraphAgentSystem compatibility!


In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

# Define the path directly
path = "/Users/dwarak/code/pensieve-ai/backend/data/testdata"
print(f"Loading documents from: {path}")

loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loading documents from: /Users/dwarak/code/pensieve-ai/backend/data/testdata
Loaded 15 documents


In [64]:
# Create and use the contextual compression retriever
# Create components
base_retriever = SimpleRetriever()
reranker = SimpleCohereReranker(model="rerank-v3.5", top_k=3)

# Create compression retriever with proper initialization
compression_retriever = SimpleCompressionRetriever(
    base_retriever=base_retriever, 
    reranker=reranker
)






In [65]:
# VectorDBService Wrapper for LangChain BaseRetriever
import asyncio
from services.vector_db import VectorDBService

class VectorDBRetriever(BaseRetriever):
    """Wrapper for VectorDBService to make it compatible with LangChain BaseRetriever"""
    
    vector_db_service: VectorDBService = Field(description="VectorDBService instance")
    limit: int = Field(default=10, description="Number of documents to retrieve")
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        """Get relevant documents from VectorDBService"""
        try:
            # Run the async search in a sync context
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            try:
                # Initialize and search
                loop.run_until_complete(self.vector_db_service.initialize())
                search_result = loop.run_until_complete(
                    self.vector_db_service.search_documents(query, limit=self.limit)
                )
            finally:
                loop.close()
            
            # Convert VectorDBService results to LangChain Documents
            documents = []
            if search_result.get("success", False):
                for doc_data in search_result.get("documents", []):
                    document = Document(
                        page_content=doc_data.get("content", ""),
                        metadata={
                            **doc_data.get("metadata", {}),
                            "score": doc_data.get("score", 0.0),
                            "id": doc_data.get("id", "")
                        }
                    )
                    documents.append(document)
            
            return documents
            
        except Exception as e:
            print(f"Error in VectorDBRetriever: {e}")
            return []

print("✅ VectorDBRetriever wrapper created!")


✅ VectorDBRetriever wrapper created!


In [82]:
# Create VectorDBService-based Compression Retriever
# Create VectorDBService instance
vector_db_service = VectorDBService()

# Create VectorDBRetriever wrapper
vector_db_retriever = VectorDBRetriever(
    vector_db_service=vector_db_service,
    limit=10  # Get top 10 documents from vector DB
)

# Create Cohere reranker
cohere_reranker = SimpleCohereReranker(model="rerank-v3.5", top_k=5)

# Create compression retriever with VectorDBService as base retriever
compression_retriever = SimpleCompressionRetriever(
    base_retriever=vector_db_retriever,
    reranker=cohere_reranker
)

print("✅ VectorDBService-based Compression Retriever created!")
print("Components:")
print("- Base Retriever: VectorDBRetriever (wraps VectorDBService)")
print("- Reranker: SimpleCohereReranker (Cohere rerank-v3.5)")
print("- Final Retriever: SimpleCompressionRetriever")
print("\nThis will:")
print("1. Search your vector database using embeddings")
print("2. Rerank the results using Cohere's rerank model")
print("3. Return the top-k most relevant documents")


✅ VectorDBService-based Compression Retriever created!
Components:
- Base Retriever: VectorDBRetriever (wraps VectorDBService)
- Reranker: SimpleCohereReranker (Cohere rerank-v3.5)
- Final Retriever: SimpleCompressionRetriever

This will:
1. Search your vector database using embeddings
2. Rerank the results using Cohere's rerank model
3. Return the top-k most relevant documents


In [67]:
# Test the VectorDBService-based Compression Retriever
query = "What are the main topics discussed in the meetings?"

print(f"Query: {query}")
print("\nResults from VectorDBService + Cohere Compression:")

try:
    results = compression_retriever._get_relevant_documents(query)
    
    if results:
        print(f"\nFound {len(results)} relevant documents:")
        for i, doc in enumerate(results, 1):
            print(f"\n{i}. Document ID: {doc.metadata.get('id', 'Unknown')}")
            print(f"   Source: {doc.metadata.get('source', 'Unknown')}")
            print(f"   Vector Score: {doc.metadata.get('score', 'N/A')}")
            print(f"   Cohere Relevance Score: {doc.metadata.get('relevance_score', 'N/A')}")
            print(f"   Content Preview: {doc.page_content[:200]}...")
    else:
        print("No documents found. Make sure your vector database has documents and your API keys are set.")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure Qdrant is running: docker run -p 6333:6333 qdrant/qdrant")
    print("2. Set your Cohere API key: os.environ['COHERE_API_KEY'] = 'your-key'")
    print("3. Make sure you have documents in your vector database")
    print("4. Check that your VectorDBService is properly initialized")


Query: What are the main topics discussed in the meetings?

Results from VectorDBService + Cohere Compression:
Error in reranking: BaseCohere.rerank() got an unexpected keyword argument 'top_k'. Did you mean 'top_n'?

Found 5 relevant documents:

1. Document ID: 3a2ea284-44e9-40fd-9f32-945eb69b97d0
   Source: Unknown
   Vector Score: 0.8231556
   Cohere Relevance Score: 1.0
   Content Preview: This is a test meeting about project planning. We discussed timelines, resources, and next steps for the Q4 launch....

2. Document ID: 62902a71-ee08-4012-9666-219b000ef0d6
   Source: Unknown
   Vector Score: 0.8230969
   Cohere Relevance Score: 1.0
   Content Preview: This is a test meeting about project planning. We discussed timelines, resources, and next steps for the Q4 launch....

3. Document ID: 71f8075f-ea9f-41e7-981d-ef2ea0074d3c
   Source: Unknown
   Vector Score: 0.8230969
   Cohere Relevance Score: 1.0
   Content Preview: This is a test meeting about project planning. We discussed tim

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/dwarak/code/pensieve-ai/backend/notebooks/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/dwarak/code/pensieve-ai/backend/notebooks/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/dwarak/code/pensieve-ai/backend/notebooks/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [7]:
# Fix Python path to import backend modules
import sys
import os

# Add the backend directory to Python path
backend_dir = "/Users/dwarak/code/pensieve-ai/backend"
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

print(f"Added {backend_dir} to Python path")
print(f"Current working directory: {os.getcwd()}")
print(f"Python path includes: {[p for p in sys.path if 'pensieve' in p]}")


Added /Users/dwarak/code/pensieve-ai/backend to Python path
Current working directory: /Users/dwarak/code/pensieve-ai/backend/notebooks
Python path includes: ['/Users/dwarak/code/pensieve-ai/backend/notebooks/.venv/lib/python3.13/site-packages', '/Users/dwarak/code/pensieve-ai/backend']


In [8]:
# Fix the import issue - use direct import since we're already in the backend directory
from agents.graph_agent import GraphAgentSystem

# Initialize the GraphAgentSystem
graph_agent = GraphAgentSystem()

print("✅ GraphAgentSystem imported and initialized successfully")
print(f"Agent type: {type(graph_agent).__name__}")


✅ GraphAgentSystem imported and initialized successfully
Agent type: GraphAgentSystem


In [10]:
# Check all ingested documents in the vector database
from services.vector_db import VectorDBService
import asyncio

async def check_ingested_documents():
    """Check all documents in the vector database"""
    try:
        # Initialize vector database service
        vector_db = VectorDBService()
        await vector_db.initialize()
        
        print("🔍 Checking ingested documents...")
        
        # Get collection statistics
        stats = await vector_db.get_collection_stats()
        print(f"📊 Collection Stats: {stats}")
        
        # Get all documents (this might be a lot if you have many)
        # Note: This is a simple approach - in production you'd want pagination
        all_docs = await vector_db.search_documents(
            query="",  # Empty query to get all
            limit=100  # Adjust limit as needed
        )
        
        print(f"📄 Found {len(all_docs.get('documents', []))} documents")
        
        # Display document details
        for i, doc in enumerate(all_docs.get('documents', []), 1):
            print(f"\n📝 Document {i}:")
            print(f"  Content: {doc.get('content', '')[:100]}...")
            print(f"  Metadata: {doc.get('metadata', {})}")
            print(f"  Score: {doc.get('score', 'N/A')}")
        
        return all_docs
        
    except Exception as e:
        print(f"❌ Error checking documents: {e}")
        return None

# Run the check
ingested_docs = await check_ingested_documents()


🔍 Checking ingested documents...
📊 Collection Stats: {'success': True, 'count': 152, 'collection_name': 'pensieve_documents', 'engine': 'qdrant'}
📄 Found 100 documents

📝 Document 1:
  Content: This is a test meeting about project planning. We discussed timelines, resources, and next steps for...
  Metadata: {'title': 'Q4 Planning Meeting', 'date': '2024-01-15', 'chunk_index': 0, 'total_chunks': 1, 'timestamp': '2025-10-20T19:57:52.110936', 'type': 'general'}
  Score: 0.68838674

📝 Document 2:
  Content: This is a test meeting about project planning. We discussed timelines, resources, and next steps for...
  Metadata: {'title': 'Q4 Planning Meeting', 'date': '2024-01-15', 'chunk_index': 0, 'total_chunks': 1, 'timestamp': '2025-10-20T19:56:51.640728', 'type': 'general'}
  Score: 0.6882892

📝 Document 3:
  Content: This is a test meeting about project planning. We discussed timelines, resources, and next steps for...
  Metadata: {'title': 'Q4 Planning Meeting', 'date': '2024-01-15', 'tim

In [23]:
# Check document ingestion status and health
async def check_ingestion_health():
    """Check the health and status of document ingestion"""
    try:
        vector_db = VectorDBService()
        await vector_db.initialize()
        
        print("🏥 Checking ingestion health...")
        
        # Get collection stats
        stats = await vector_db.get_collection_stats()
        print(f"📊 Collection Statistics:")
        print(f"  Total Documents: {stats.get('points_count', 0)}")
        print(f"  Collection Name: {stats.get('collection_name', 'Unknown')}")
        
        # Check if collection exists and is healthy
        if stats.get('points_count', 0) > 0:
            print("✅ Collection is healthy and contains documents")
        else:
            print("⚠️ Collection is empty or not initialized")
        
        # Test a simple search to verify functionality
        test_search = await vector_db.search_documents("test", limit=1)
        if test_search.get('success', False):
            print("✅ Search functionality is working")
        else:
            print("❌ Search functionality has issues")
        
        return stats
        
    except Exception as e:
        print(f"❌ Error checking ingestion health: {e}")
        return None

# Run health check
health_status = await check_ingestion_health()


🏥 Checking ingestion health...
📊 Collection Statistics:
  Total Documents: 0
  Collection Name: pensieve_documents
⚠️ Collection is empty or not initialized
✅ Search functionality is working


In [ ]:
# Test the fixed SimpleCohereReranker
print("🧪 Testing the updated SimpleCohereReranker...")

# Create a test reranker
test_reranker = SimpleCohereReranker(model="rerank-v3.5", top_k=3)

# Test documents
test_docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Python is a programming language.",
    "Natural language processing deals with human language."
]
test_query = "What is machine learning?"

try:
    test_results = test_reranker.rerank(test_query, test_docs)
    print(f"✅ Reranker test successful! Got {len(test_results)} results")
    for i, result in enumerate(test_results, 1):
        print(f"  {i}. Score: {result['relevance_score']:.3f} - {result['document'][:50]}...")
except Exception as e:
    print(f"❌ Reranker test failed: {e}")

print("\n✅ SimpleCohereReranker has been fixed with parameter compatibility!")


In [ ]:
# Update compression retriever with the fixed SimpleCohereReranker
# Create new reranker with the fixed SimpleCohereReranker
updated_reranker = SimpleCohereReranker(model="rerank-v3.5", top_k=5)

# Create updated compression retriever
updated_compression_retriever = SimpleCompressionRetriever(
    base_retriever=vector_db_retriever,
    reranker=updated_reranker
)

print("✅ Updated compression retriever with fixed SimpleCohereReranker!")
print("The SimpleCohereReranker now handles both 'top_k' and 'top_n' parameters automatically.")

# Verify the compression retriever works
print("\n🔍 Verifying compression retriever...")
if hasattr(updated_compression_retriever, 'initialize'):
    print("✅ Has initialize method")
if hasattr(updated_compression_retriever, 'search_documents'):
    print("✅ Has search_documents method")
if hasattr(updated_compression_retriever, '_get_relevant_documents'):
    print("✅ Has _get_relevant_documents method")

print("\n🎯 Ready to use with GraphAgentSystem!")


In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)



dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/15 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/15 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary' already exists in node '16917c'. Skipping!
Property 'summary' already exists in node 'ca8b0a'. Skipping!
Property 'summary' already exists in node '343f07'. Skipping!
Property 'summary' already exists in node 'fcb678'. Skipping!
Property 'summary' already exists in node '6ac434'. Skipping!
Property 'summary' already exists in node 'e07934'. Skipping!
Property 'summary' already exists in node '58d23d'. Skipping!
Property 'summary' already exists in node '19057e'. Skipping!
Property 'summary' already exists in node '8e0534'. Skipping!
Property 'summary' already exists in node '32c061'. Skipping!
Property 'summary' already exists in node '4c18f0'. Skipping!
Property 'summary' already exists in node '5fe90b'. Skipping!
Property 'summary' already exists in node '5fe2dc'. Skipping!
Property 'summary' already exists in node '41c3b4'. Skipping!
Property 'summary' already exists in node 'd78389'. Skipping!


Applying CustomNodeFilter: 0it [00:00, ?it/s]

...

Property 'summary_embedding' already exists in node '16917c'. Skipping!
Property 'summary_embedding' already exists in node '4c18f0'. Skipping!
Property 'summary_embedding' already exists in node 'd78389'. Skipping!
Property 'summary_embedding' already exists in node 'ca8b0a'. Skipping!
Property 'summary_embedding' already exists in node 'fcb678'. Skipping!
Property 'summary_embedding' already exists in node '8e0534'. Skipping!
Property 'summary_embedding' already exists in node '343f07'. Skipping!
Property 'summary_embedding' already exists in node '6ac434'. Skipping!
Property 'summary_embedding' already exists in node 'e07934'. Skipping!
Property 'summary_embedding' already exists in node '58d23d'. Skipping!
Property 'summary_embedding' already exists in node '5fe90b'. Skipping!
Property 'summary_embedding' already exists in node '5fe2dc'. Skipping!
Property 'summary_embedding' already exists in node '19057e'. Skipping!
Property 'summary_embedding' already exists in node '32c061'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,what do i do about the project phoenix meeting...,[[MEETING] Title: Sprint Retrospective - Proje...,The context includes multiple meetings with va...,single_hop_specifc_query_synthesizer
1,What is the main focus of the meetings documen...,[[MEETING] Title: Project Triton Kickoff Time:...,The meetings cover various topics including pr...,single_hop_specifc_query_synthesizer
2,What is the main focus of the project Nebula?,[[MEETING] Title: Project Nebula - Design Revi...,The context mentions that the color palette fo...,single_hop_specifc_query_synthesizer
3,What role does the Product Development Coordin...,[[MEETING] Title: Project Atlas - Data Ingesti...,The Product Development Coordinator coordinate...,single_hop_specifc_query_synthesizer
4,What are the key responsibilities of a Product...,[[TODO] Review Sarah's updated flow diagrams f...,The Product Development Coordinator coordinate...,single_hop_specifc_query_synthesizer


In [ ]:
''' Fetch all the meeting notes '''


''' Ingest them into Vector DB and in  persistent store '''
from services.document_ingestion import DocumentIngestionService
document_ingestion = DocumentIngestionService()
print(docs)
for d in docs:
    await document_ingestion.ingest_document(d.page_content, d.metadata)


[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/Users/dwarak/code/pensieve-ai/backend/data/testdata/meetingnotes.pdf', 'file_path': '/Users/dwarak/code/pensieve-ai/backend/data/testdata/meetingnotes.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': 'meetingnotes', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content="[MEETING] Title: Sprint Retrospective - Project Phoenix Time: 10/21/2025, 12:16:55 PM \nAttendees: Alex Johnson, $YOURS_TRULY Notes: [NOTE] The team felt the last sprint was \noverloaded. Alex agreed to review the velocity calculation process with the team next week. \nDeployment was smoother than expected thanks to the new automated script. Alex mentioned \na potential bottleneck in the API documentation review process. [ENDNOTE] [FEEDBACK] FOR \nAlex Johnson FROM $YOURS_TRULY Great job running the post-mortem, very \n

NameError: name 'check_ingestion_health' is not defined

In [14]:

''' Initialize agents and test sample queries '''

async def test_graph_agent():
    """Test the GraphAgentSystem with a sample query"""
    try:
        # Sample query
        query = "What is the latest about Project Phoenix"
        context = "Meeting"
        
        print(f"🔍 Testing query: {query}")
        print(f"📝 Context: {context}")
        
        # Process the query
        result = await graph_agent.process_query(query, context)
        
        print(f"\n📊 Results:")
        print(f"Success: {result.get('success', False)}")
        print(f"Response: {result.get('response', 'No response')}")
        print(f"Sources: {result.get('sources', {})}")
        
        if result.get('analysis'):
            analysis = result['analysis']
            print(f"Analysis: {analysis}")
        
        return result
        
    except Exception as e:
        print(f"❌ Error testing GraphAgentSystem: {e}")
        return None

# Run the test
result = await test_graph_agent()





''' Run the agent and hook langsmith'''


''' Fetch and publish langsmith results '''

🔍 Testing query: What is the latest about Project Phoenix
📝 Context: Meeting


⚠️ SERP_API_KEY not configured, using mock search results



📊 Results:
Success: True
Response: Based on the meeting notes, the latest updates on Project Phoenix are as follows:

1. During the Sprint Retrospective, the team expressed that the last sprint was overloaded. In response, Alex Johnson agreed to take certain measures, although the specific actions weren't detailed in the notes.

2. In the Bug Triage Session, a Priority 1 (P1) bug was identified, titled 'User Profile Load Failure'. However, the notes do not provide further information about the resolution or steps taken towards fixing this bug.

Unfortunately, the notes lack detailed information about the actions Alex Johnson agreed to take or the resolution of the P1 bug. For more specific updates on these matters, consider checking more recent meeting notes or directly contacting the attendees.
Sources: {'meeting_notes': 5, 'web_results': 0}
Analysis: {'needs_meeting_search': True, 'needs_web_search': True, 'search_strategy': 'A comprehensive search is required as the query is asking

' Fetch and publish langsmith results '

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8 - Certification Challenge -2"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [18]:
eval_llm = ChatOpenAI(model="gpt-4.1")

In [19]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

ModuleNotFoundError: No module named 'langchain.evaluation'

In [32]:
for test_row in dataset:
  response = await graph_agent.process_query(test_row.eval_sample.user_input, "Meeting")
  print(response)
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.content for context in response["meeting_results"]]


{'success': True, 'response': 'Your query covers a very wide range of topics. To provide the most helpful response, could you please clarify which of these areas you\'re most interested in at the moment? Here are a few possible ways I can assist:\n\n1. If you are looking for specific information about "Project Phoenix", I can provide details based on the meeting notes available.\n2. If you need to address an API bottleneck or solve an authentication issue, I can summarize the relevant meeting notes or pull up potential solutions based on web search results.\n3. If you want to know about the deployment update, the team feedback, or the project progress, I can highlight these points from the meeting notes.\n4. If you\'re interested in discussing the next steps, the project scope, or the certification goals, I can summarize discussions on these topics from the meeting notes.\n5. If you want to review the budget, the candidate review, or team updates, I can provide the necessary informatio

⚠️ SERP_API_KEY not configured, using mock search results


{'success': True, 'response': "I'm sorry, but I cannot provide the specific responsibilities of a Product Development Coordinator in ensuring successful product deployment as I don't have relevant meeting notes or web search results to reference. Could you provide more information or context?", 'meeting_results': [], 'web_results': [], 'sources': {'meeting_notes': 0, 'web_results': 0}, 'analysis': {'needs_meeting_search': False, 'needs_web_search': True, 'search_strategy': 'A web search is needed to gather information about the key responsibilities of a Product Development Coordinator in ensuring successful product deployment. This is not meeting-specific information, but rather general knowledge about a specific job role.', 'priority': 'web_search'}}


In [33]:
dataset.samples[0].eval_sample.response

'Your query covers a very wide range of topics. To provide the most helpful response, could you please clarify which of these areas you\'re most interested in at the moment? Here are a few possible ways I can assist:\n\n1. If you are looking for specific information about "Project Phoenix", I can provide details based on the meeting notes available.\n2. If you need to address an API bottleneck or solve an authentication issue, I can summarize the relevant meeting notes or pull up potential solutions based on web search results.\n3. If you want to know about the deployment update, the team feedback, or the project progress, I can highlight these points from the meeting notes.\n4. If you\'re interested in discussing the next steps, the project scope, or the certification goals, I can summarize discussions on these topics from the meeting notes.\n5. If you want to review the budget, the candidate review, or team updates, I can provide the necessary information from the meetings.\n6. If yo

In [34]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [35]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

In [36]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[29]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')


{'context_recall': 0.2889, 'faithfulness': 0.6210, 'factual_correctness': 0.3160, 'answer_relevancy': 0.1906, 'context_entity_recall': 0.2700, 'noise_sensitivity_relevant': 0.2724}

In [83]:
import importlib
import sys
# Reload specific modules
if 'agents.graph_agent' in sys.modules:
    importlib.reload(sys.modules['agents.graph_agent'])

if 'services.vector_db' in sys.modules:
    importlib.reload(sys.modules['services.vector_db'])

# Now reimport
from agents.graph_agent import GraphAgentSystem


# Create your compression retriever (VectorDBService + Cohere reranking)
compression_retriever = SimpleCompressionRetriever(
    base_retriever=vector_db_retriever,
    reranker=cohere_reranker
)

# Create GraphAgentSystem
reranked_graph = GraphAgentSystem(retriever=compression_retriever)

# Set your compression retriever using set_retriever()
reranked_graph.set_retriever(compression_retriever)
for test_row in dataset:
  response = await reranked_graph.process_query(test_row.eval_sample.user_input, "Meeting")
  print(response)
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.content for context in response["meeting_results"]]

✅ Compression retriever initialized successfully
{'success': True, 'response': "I'm sorry, but your query is extensive and covers multiple topics. I don't have enough context from the provided meeting notes to answer all parts of your question. The meeting notes you provided only mention a sprint retrospective for Project Phoenix and feedback regarding the team feeling the last sprint was overloaded. \n\nCould you please specify what information you need about each topic? For example, are you looking for a summary of the meeting notes, solutions for an API bottleneck, updates on deployment, feedback from the team, or next steps for Project Phoenix? If you could provide more detailed questions or more context, I'd be able to assist you better.", 'meeting_results': [SearchResult(content="[MEETING] Title: Sprint Retrospective - Project Phoenix Time: 10/21/2025, 12:16:55 PM \nAttendees: Alex Johnson, $YOURS_TRULY Notes: [NOTE] The team felt the last sprint was \noverloaded. Alex agreed to 

⚠️ SERP_API_KEY not configured, using mock search results


{'success': True, 'response': "I'm afraid I can't provide a specific answer without more context from the meeting notes. However, in general, a Product Development Coordinator's key responsibilities in ensuring successful product deployment could include:\n\n1. **Cross-Functional Coordination**: They need to communicate effectively with various departments (such as engineering, design, marketing, and sales) to ensure that all aspects of the product are aligned.\n\n2. **Schedule Management**: They need to oversee the product development timeline, ensuring that all milestones are met and that the product can be deployed on schedule.\n\n3. **Quality Assurance**: They need to work closely with the quality assurance (QA) team to ensure the product meets the necessary standards and specifications.\n\n4. **Market Research**: They need to understand the market and customer needs to align the product effectively.\n\n5. **Product Testing**: They need to coordinate testing phases to identify and 

In [84]:
evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [85]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[29]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')


{'context_recall': 0.1889, 'faithfulness': 0.3654, 'factual_correctness': 0.4120, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.2726, 'noise_sensitivity_relevant': 0.4689}